# Organize the references into a single format

In [22]:
import pandas as pd
import os


In [23]:

cwd = os.getcwd()
if cwd.endswith("notebooks"):
    os.chdir("..")
    print(f"Changed working directory to {os.getcwd()}")

In [24]:
OT_parallel_df = pd.read_csv("data/Biblical Parallel Passages Table - Biblical Parallel Passages Table.csv")
NT_parallel_df = pd.read_csv("data/Synoptic Gospel Parallels - Synoptic Gospels Comparison.csv")
NT_OT_quotes_df = pd.read_csv("data/truncated_references.csv")

In [25]:
import pandas as pd
from itertools import combinations

# Columns that contain passage references
gospel_cols = ["Matthew", "Mark", "Luke", "John"]

# Optional cleanup (your sample includes a repeated header row inside the data)
df = NT_parallel_df.copy()
df = df[df["Pericope"] != "Pericope"].copy()
df[gospel_cols] = df[gospel_cols].replace(r"^\s*$", pd.NA, regex=True)

pair_rows = []

for _, row in df.iterrows():
    pericope = row["Pericope"]

    # Keep only non-null gospel references in this row
    present = [(g, row[g]) for g in gospel_cols if pd.notna(row[g])]

    # If 2 refs -> 1 pair; if 3 refs -> 3 pairs; if 4 refs -> 6 pairs
    for (g1, r1), (g2, r2) in combinations(present, 2):
        pair_rows.append({
            "Ref_1": g1 + " " + r1,
            "Ref_2": g2 + " " + r2
        })

NT_pairs_df = pd.DataFrame(pair_rows)

print(f"Total pair rows: {len(NT_pairs_df)}")
NT_pairs_df.head(20)


Total pair rows: 305


,Ref_1,Ref_2
0,Matthew 1:1,Mark 1:1
1,Matthew 1:1,Luke 1:1-4
2,Matthew 1:1,John 1:1-18
3,Mark 1:1,Luke 1:1-4
4,Mark 1:1,John 1:1-18
5,Luke 1:1-4,John 1:1-18
6,Matthew 1:2-17,Luke 3:23-38
7,Matthew 1:18-25,Luke 2:1-7
8,Matthew 2:1-12,Luke 2:8-20
9,Matthew 2:22-23,Luke 2:39-40


In [26]:
from itertools import combinations

# Columns that contain OT parallel references
ot_ref_cols = [
    "Samuel / Kings Reference",
    "Chronicles Reference",
    "Isaiah Reference",
]

ot_df = OT_parallel_df.copy()
ot_df[ot_ref_cols] = ot_df[ot_ref_cols].replace(r"^\s*$", pd.NA, regex=True)

ot_pair_rows = []

for idx, row in ot_df.iterrows():
    present = [(col, row[col]) for col in ot_ref_cols if pd.notna(row[col])]

    # If 2 refs -> 1 pair; if 3 refs -> 3 pairs
    for (c1, r1), (c2, r2) in combinations(present, 2):
        ot_pair_rows.append({
            "Ref_1": f"{r1}",
            "Ref_2": f"{r2}",
        })

OT_pairs_df = pd.DataFrame(ot_pair_rows)

print(f"Total OT pair rows: {len(OT_pairs_df)}")
OT_pairs_df.head(20)

Total OT pair rows: 121


,Ref_1,Ref_2
0,1 Samuel 27,1 Chronicles 12:1-7
1,1 Samuel 29:1-3,1 Chronicles 12:19-22
2,1 Samuel 31,1 Chronicles 10
3,2 Samuel 5:1-5,1 Chronicles 11:1-3
4,2 Samuel 5:6-10,1 Chronicles 11:4-9
5,2 Samuel 5:11-16,1 Chronicles 14:1-7
6,2 Samuel 5:17-25,1 Chronicles 14:8-17
7,2 Samuel 6:1-11,1 Chronicles 13
8,2 Samuel 6:12-23,1 Chronicles 15 & 16
9,2 Samuel 7,1 Chronicles 17


In [27]:
NT_OT_pairs_df = NT_OT_quotes_df.drop('LXX (Brenton)', axis=1)
NT_OT_pairs_df = NT_OT_pairs_df.rename(columns={'New Testament (AV)': 'Ref_1', 'Masoretic (AV)': 'Ref_2'})
NT_OT_pairs_df.head()

,Ref_1,Ref_2
0,Matthew 1:23,Isaiah 7:14
1,Matthew 2:6,Micah 5:2
2,Matthew 2:15,Hosea 11:1
3,Matthew 2:18,Jeremiah 31:15
4,Matthew 3:3,Isaiah 40:3-5


In [28]:
all_pairs_df = pd.concat([NT_OT_pairs_df, NT_pairs_df, OT_pairs_df])
print(len(all_pairs_df))
print(all_pairs_df.head(20))

616
               Ref_1                 Ref_2
0       Matthew 1:23           Isaiah 7:14
1        Matthew 2:6             Micah 5:2
2       Matthew 2:15            Hosea 11:1
3       Matthew 2:18        Jeremiah 31:15
4        Matthew 3:3         Isaiah 40:3-5
5        Matthew 4:4       Deuteronomy 8:3
6        Matthew 4:6       Psalms 91:11,12
7        Matthew 4:7      Deuteronomy 6:16
8       Matthew 4:10  Deuteronomy 6:13, 10
9    Matthew 4:15,16          Isaiah 9:1,2
10      Matthew 8:17           Isaiah 53:4
11      Matthew 9:13             Hosea 6:6
12  Matthew 10:35,36             Micah 7:6
13     Matthew 11:10           Malachi 3:1
14  Matthew 12:18-21         Isaiah 42:1-4
15  Matthew 13:14,15         Isaiah 6:9,10
16     Matthew 13:35           Psalms 78:2
17      Matthew 15:4          Exodus 20:12
18      Matthew 15:4          Exodus 21:17
19    Matthew 15:8,9          Isaiah 29:13


# Pull the Greek references and explode longer passages to represent multiple lines

In [29]:
url = "https://huggingface.co/datasets/hmcgovern/original-language-bibles-greek/resolve/main/data/train-00000-of-00001.parquet"
NT_df = pd.read_parquet(url)


In [58]:
ot_df = pd.read_parquet("hf://datasets/epaolinos/septuagint/data/train-00000-of-00001-1ac4cc64c0bad5a7.parquet")


In [59]:
ot_df['Book'].unique()

array(['ΓΕΝΕΣΙΣ', 'ΕΞΟΔΟΣ', 'ΛΕΥΙΤΙΚΟΝ', 'ΑΡΙΘΜΟΙ', 'ΔΕΥΤΕΡΟΝΟΜΙΟΝ',
       'ΙΗΣΟΥΣ', 'ΚΡΙΤΑΙ (Codex Alexandrinus)',
       'ΚΡΙΤΑΙ (Codex Vaticanus)', 'ΡΟΥΘ', 'ΒΑΣΙΛΕΙΩΝ Α (M 1Sam)',
       'ΒΑΣΙΛΕΙΩΝ Β (M 2Sam)', 'ΒΑΣΙΛΕΙΩΝ Γ (M 1Regn)',
       'ΒΑΣΙΛΕΙΩΝ Δ (M 2Regn)', 'ΠΑΡΑΛΕΙΠΟΜΕΝΩΝ Α', 'ΠΑΡΑΛΕΙΠΟΜΕΝΩΝ Β',
       'ΕΣΔΡΑΣ Α (liber apocryphus)', 'ΕΣΔΡΑΣ Β (M Esdr, Neh)', 'ΕΣΘΗΡ',
       'ΙΟΥΔΙΘ', 'ΤΩΒΙΤ (Codices Alexandrinus et Vaticanus)',
       'ΤΩΒΙΤ (Codex Sinaiticus)', 'ΜΑΚΚΑΒΑΙΩΝ Α', 'ΜΑΚΚΑΒΑΙΩΝ Β',
       'ΜΑΚΚΑΒΑΙΩΝ Γ', 'ΜΑΚΚΑΒΑΙΩΝ Δ', 'ΨΑΛΜΟΙ', 'ΩΔΑΙ', 'ΠΑΡΟΙΜΙΑΙ',
       'ΕΚΚΛΗΣΙΑΣΤΗΣ', 'ΑΣΜΑ', 'ΙΩΒ', 'ΣΟΦΙΑ ΣΑΛΩΜΩΝΟΣ', 'ΣΟΦΙΑ ΣΕΙΡΑΧ',
       'ΨΑΛΜΟΙ ΣΟΛΟΜΩΝΤΟΣ', 'ΩΣΗΕ', 'ΑΜΩΣ', 'ΜΙΧΑΙΑΣ', 'ΙΩΗΛ', 'ΑΒΔΙΟΥ',
       'ΙΩΝΑΣ', 'ΝΑΟΥΜ', 'ΑΜΒΑΚΟΥΜ', 'ΣΟΦΟΝΙΑΣ', 'ΑΓΓΑΙΟΣ', 'ΖΑΧΑΡΙΑΣ',
       'ΜΑΛΑΧΙΑΣ', 'ΗΣΑΙΑΣ', 'ΙΕΡΕΜΙΑΣ', 'ΒΑΡΟΥΧ', 'ΘΡΗΝΟΙ',
       'ΕΠΙΣΤΟΛΗ ΙΕΡΕΜΙΟΥ', 'ΙΕΖΕΚΙΗΛ', 'ΣΟΥΣΑΝΝΑ',
       'ΣΟΥΣΑΝΝΑ (Θεοδοτίων)', 'ΔΑΝΙΗΛ (LXX)', 'ΔΑΝΙΗΛ (Θεοδοτίων)',
       'ΒΗΛ ΚΑΙ 

In [60]:
lxx_mapping = {
    'ΓΕΝΕΣΙΣ': 'Genesis',
    'ΕΞΟΔΟΣ': 'Exodus',
    'ΛΕΥΙΤΙΚΟΝ': 'Leviticus',
    'ΑΡΙΘΜΟΙ': 'Numbers',
    'ΔΕΥΤΕΡΟΝΟΜΙΟΝ': 'Deuteronomy',
    'ΙΗΣΟΥΣ': 'Joshua',
    'ΚΡΙΤΑΙ (Codex Alexandrinus)': 'Judges (Alexandrinus)',
    'ΚΡΙΤΑΙ (Codex Vaticanus)': 'Judges (Vaticanus)',
    'ΡΟΥΘ': 'Ruth',
    'ΒΑΣΙΛΕΙΩΝ Α (M 1Sam)': '1 Samuel',
    'ΒΑΣΙΛΕΙΩΝ Β (M 2Sam)': '2 Samuel',
    'ΒΑΣΙΛΕΙΩΝ Γ (M 1Regn)': '1 Kings',
    'ΒΑΣΙΛΕΙΩΝ Δ (M 2Regn)': '2 Kings',
    'ΠΑΡΑΛΕΙΠΟΜΕΝΩΝ Α': '1 Chronicles',
    'ΠΑΡΑΛΕΙΠΟΜΕΝΩΝ Β': '2 Chronicles',
    'ΕΣΔΡΑΣ Α (liber apocryphus)': '1 Esdras',
    'ΕΣΔΡΑΣ Β (M Esdr, Neh)': 'Ezra-Nehemiah',
    'ΕΣΘΗΡ': 'Esther',
    'ΙΟΥΔΙΘ': 'Judith',
    'ΤΩΒΙΤ (Codices Alexandrinus et Vaticanus)': 'Tobit (Alex/Vat)',
    'ΤΩΒΙΤ (Codex Sinaiticus)': 'Tobit (Sinaiticus)',
    'ΜΑΚΚΑΒΑΙΩΝ Α': '1 Maccabees',
    'ΜΑΚΚΑΒΑΙΩΝ Β': '2 Maccabees',
    'ΜΑΚΚΑΒΑΙΩΝ Γ': '3 Maccabees',
    'ΜΑΚΚΑΒΑΙΩΝ Δ': '4 Maccabees',
    'ΨΑΛΜΟΙ': 'Psalms',
    'ΩΔΑΙ': 'Odes',
    'ΠΑΡΟΙΜΙΑΙ': 'Proverbs',
    'ΕΚΚΛΗΣΙΑΣΤΗΣ': 'Ecclesiastes',
    'ΑΣΜΑ': 'Song of Solomon',
    'ΙΩΒ': 'Job',
    'ΣΟΦΙΑ ΣΑΛΩΜΩΝΟΣ': 'Wisdom of Solomon',
    'ΣΟΦΙΑ ΣΕΙΡΑΧ': 'Sirach',
    'ΨΑΛΜΟΙ ΣΟΛΟΜΩΝΤΟΣ': 'Psalms of Solomon',
    'ΩΣΗΕ': 'Hosea',
    'ΑΜΩΣ': 'Amos',
    'ΜΙΧΑΙΑΣ': 'Micah',
    'ΙΩΗΛ': 'Joel',
    'ΑΒΔΙΟΥ': 'Obadiah',
    'ΙΩΝΑΣ': 'Jonah',
    'ΝΑΟΥΜ': 'Nahum',
    'ΑΜΒΑΚΟΥΜ': 'Habakkuk',
    'ΣΟΦΟΝΙΑΣ': 'Zephaniah',
    'ΑΓΓΑΙΟΣ': 'Haggai',
    'ΖΑΧΑΡΙΑΣ': 'Zechariah',
    'ΜΑΛΑΧΙΑΣ': 'Malachi',
    'ΗΣΑΙΑΣ': 'Isaiah',
    'ΙΕΡΕΜΙΑΣ': 'Jeremiah',
    'ΒΑΡΟΥΧ': 'Baruch',
    'ΘΡΗΝΟΙ': 'Lamentations',
    'ΕΠΙΣΤΟΛΗ ΙΕΡΕΜΙΟΥ': 'Letter of Jeremiah',
    'ΙΕΖΕΚΙΗΛ': 'Ezekiel',
    'ΣΟΥΣΑΝΝΑ': 'Susanna',
    'ΣΟΥΣΑΝΝΑ (Θεοδοτίων)': 'Susanna (Theodotion)',
    'ΔΑΝΙΗΛ (LXX)': 'Daniel (LXX)',
    'ΔΑΝΙΗΛ (Θεοδοτίων)': 'Daniel (Theodotion)',
    'ΒΗΛ ΚΑΙ ΔΡΑΚΩΝ': 'Bel and the Dragon'
}

In [61]:
ot_df['Book_english'] = ot_df['Book'].replace(lxx_mapping)

In [62]:
ot_df['Book_english'].unique()

array(['Genesis', 'Exodus', 'Leviticus', 'Numbers', 'Deuteronomy',
       'Joshua', 'Judges (Alexandrinus)', 'Judges (Vaticanus)', 'Ruth',
       '1 Samuel', '2 Samuel', '1 Kings', '2 Kings', '1 Chronicles',
       '2 Chronicles', '1 Esdras', 'Ezra-Nehemiah', 'Esther', 'Judith',
       'Tobit (Alex/Vat)', 'Tobit (Sinaiticus)', '1 Maccabees',
       '2 Maccabees', '3 Maccabees', '4 Maccabees', 'Psalms', 'Odes',
       'Proverbs', 'Ecclesiastes', 'Song of Solomon', 'Job',
       'Wisdom of Solomon', 'Sirach', 'Psalms of Solomon', 'Hosea',
       'Amos', 'Micah', 'Joel', 'Obadiah', 'Jonah', 'Nahum', 'Habakkuk',
       'Zephaniah', 'Haggai', 'Zechariah', 'Malachi', 'Isaiah',
       'Jeremiah', 'Baruch', 'Lamentations', 'Letter of Jeremiah',
       'Ezekiel', 'Susanna', 'Susanna (Theodotion)', 'Daniel (LXX)',
       'Daniel (Theodotion)', 'Bel and the Dragon'], dtype=object)

In [63]:
ot_df['reference'] = ot_df['Book_english'].astype(str) + " " + ot_df['Chapter'].astype(str) + ":" + ot_df['Verse Number'].astype(str)
ot_df['text'] = ot_df['Verse Text']

ot_df.drop(columns=['Book', 'Chapter', 'Verse Number', 'Verse Text', 'Genre', 'Book_english'], inplace=True)

In [64]:
NT_df.head()

,reference,text,transliteration,translation,dStrongs,manuscript_source
0,Mat.1.1.01,Βίβλος,Biblos,[The] book,G0976=N-NSF,NKO
1,Mat.1.1.02,γενέσεως,geneseōs,of [the] genealogy,G1078=N-GSF,NKO
2,Mat.1.1.03,Ἰησοῦ,Iēsou,of Jesus,G2424G=N-GSM-P,NKO
3,Mat.1.1.04,Χριστοῦ,Christou,Christ,G5547=N-GSM-T,NKO
4,Mat.1.1.05,υἱοῦ,huiou,son,G5207=N-GSM,NKO


In [65]:
parts = NT_df["reference"].astype(str).str.extract(
    r"^(?P<book>[^.]+)\.(?P<chapter>\d+)\.(?P<verse>\d+)(?:[\[\{\(][^\]\}\)]+[\]\}\)])?\.(?P<word>\d+)$"
)


tmp = NT_df.join(parts)

# Keep only rows where reference parsed correctly
tmp = tmp.dropna(subset=["book", "chapter", "verse", "word"]).copy()

# Safe numeric conversion
tmp["chapter"] = pd.to_numeric(tmp["chapter"], errors="coerce")
tmp["verse"] = pd.to_numeric(tmp["verse"], errors="coerce")
tmp["word"] = pd.to_numeric(tmp["word"], errors="coerce")
tmp = tmp.dropna(subset=["chapter", "verse", "word"]).copy()

tmp["chapter"] = tmp["chapter"].astype(int)
tmp["verse"] = tmp["verse"].astype(int)
tmp["word"] = tmp["word"].astype(int)

tmp["verse_ref"] = (
    tmp["book"] + "." + tmp["chapter"].astype(str) + "." + tmp["verse"].astype(str)
)

verse_df = (
    tmp.sort_values(["book", "chapter", "verse", "word"])
       .groupby("verse_ref", as_index=False, sort=False)
       .agg(
           text=("text", lambda s: " ".join(s.astype(str)))
       )
)

# print(verse_df.head(10))

# Optional: inspect bad references that failed parsing
bad_refs = NT_df.loc[parts.isna().any(axis=1), "reference"]
print("Unparsed refs sample:", bad_refs.head(20).tolist())


Unparsed refs sample: ['._Word']


In [66]:
verse_df.head(10)

,verse_ref,text
0,1Co.1.1,Παῦλος κλητὸς ἀπόστολος Χριστοῦ Ἰησοῦ διὰ θελή...
1,1Co.1.2,"τῇ ἐκκλησίᾳ τοῦ θεοῦ τῇ οὔσῃ ἐν Κορίνθῳ, ἡγι..."
2,1Co.1.3,χάρις ὑμῖν καὶ εἰρήνη ἀπὸ θεοῦ πατρὸς ἡμῶν καὶ...
3,1Co.1.4,Εὐχαριστῶ τῷ θεῷ μου πάντοτε περὶ ὑμῶν ἐπὶ τῇ...
4,1Co.1.5,ὅτι ἐν παντὶ ἐπλουτίσθητε ἐν αὐτῷ ἐν παντὶ λόγ...
5,1Co.1.6,"καθὼς τὸ μαρτύριον τοῦ Χριστοῦ ἐβεβαιώθη ἐν ὑμῖν,"
6,1Co.1.7,ὥστε ὑμᾶς μὴ ὑστερεῖσθαι ἐν μηδενὶ χαρίσματι ἀ...
7,1Co.1.8,ὃς καὶ βεβαιώσει ὑμᾶς ἕως τέλους ἀνεγκλήτους ἐ...
8,1Co.1.9,"Πιστὸς ὁ θεός, δι᾽ οὗ ἐκλήθητε εἰς κοινωνίαν τ..."
9,1Co.1.10,"Παρακαλῶ δὲ ὑμᾶς, ἀδελφοί, διὰ τοῦ ὀνόματος το..."


In [67]:
import pandas as pd
import re

def split_complex_refs(text):
    if not isinstance(text, str): return [text]
    
    # 1. Capture the book name (e.g., "2 Kings " or "Matthew ")
    # Matches optional digit + space + word + space
    match = re.match(r'^(\d?\s?[a-zA-Z]+\s+)', text)
    if not match: return [text]
    book_prefix = match.group(1)

    # 2. Split on either:
    # - A semicolon followed by optional space
    # - A space that is specifically between two digits
    parts = re.split(r';\s*|(?<=\d)\s+(?=\d)', text)
    
    # 3. Clean and re-attach the book name if missing
    result = []
    for p in parts:
        p = p.strip()
        if not p: continue
        # If the part doesn't start with the book name, prepend it
        if not p.startswith(book_prefix.strip()):
            result.append(f"{book_prefix}{p}")
        else:
            result.append(p)
    return result

# Apply and explode
all_pairs_df['Ref_1'] = all_pairs_df['Ref_1'].apply(split_complex_refs)
all_pairs_df = all_pairs_df.explode('Ref_1').reset_index(drop=True)
# Apply and explode for Ref_2 as well
all_pairs_df['Ref_2'] = all_pairs_df['Ref_2'].apply(split_complex_refs)
all_pairs_df = all_pairs_df.explode('Ref_2').reset_index(drop=True)

In [68]:
all_pairs_df.head()

,Ref_1,Ref_2
0,matthew 1:23,isaiah 7:14
1,matthew 2:6,micah 5:2
2,matthew 2:15,hosea 11:1
3,matthew 2:18,jeremiah 31:15
4,matthew 3:3,isaiah 40:3


In [69]:
def expand_verse_ranges(text):
    if not isinstance(text, str) or ':' not in text:
        return [text]
    
    # 1. Separate Prefix and Verse part
    match = re.match(r'^(.*:)(.*)$', text)
    if not match:
        return [text]
    
    prefix = match.group(1) 
    verse_part = match.group(2)
    
    # 2. Split by comma
    parts = [p.strip() for p in verse_part.split(',')]
    
    expanded_verses = []
    for p in parts:
        # Remove any non-numeric characters except the hyphen (e.g., '14b' -> '14')
        clean_p = re.sub(r'[^\d\-]', '', p)
        if not clean_p: continue

        if '-' in clean_p:
            try:
                start_str, end_str = clean_p.split('-', 1)
                start, end = int(start_str), int(end_str)
                expanded_verses.extend(range(start, end + 1))
            except ValueError:
                continue
        else:
            try:
                expanded_verses.append(int(clean_p))
            except ValueError:
                continue
                
    return [f"{prefix}{v}" for v in expanded_verses]

# Assuming your columns are 'Col1' and 'Col2'
# First, convert ranges into lists of individual verses
all_pairs_df['Ref_1'] = all_pairs_df['Ref_1'].apply(expand_verse_ranges)
all_pairs_df['Ref_2'] = all_pairs_df['Ref_2'].apply(expand_verse_ranges)

# Second, explode Col1. This duplicates the lists in Col2 for every verse in Col1.
all_pairs_df = all_pairs_df.explode('Ref_1')

# Third, explode Col2. This creates the Cartesian product (every verse vs every verse).
all_pairs_df = all_pairs_df.explode('Ref_2').reset_index(drop=True)

In [70]:
all_pairs_df.head()

,Ref_1,Ref_2
0,matthew 1:23,isaiah 7:14
1,matthew 2:6,micah 5:2
2,matthew 2:15,hosea 11:1
3,matthew 2:18,jeremiah 31:15
4,matthew 3:3,isaiah 40:3


In [71]:
# Common aliases that appear in your pair tables
book_aliases = {
    "matthew": "mat",
    "mark": "mrk",
    "luke": "luk",
    "john": "jhn",
    "acts": "act",
    "romans": "rom",
    "1 corinthians": "1co",
    "2 corinthians": "2co",
    "galatians": "gal",
    "ephesians": "eph",
    "philippians": "php",
    "colossians": "col",
    "1 thessalonians": "1th",
    "2 thessalonians": "2th",
    "1 timothy": "1ti",
    "2 timothy": "2ti",
    "titus": "tit",
    "philemon": "phm",
    "hebrews": "heb",
    "james": "jas",
    "1 peter": "1pe",
    "2 peter": "2pe",
    "1 john": "1jn",
    "2 john": "2jn", 
    "3 john": "3jn",
    "jude": "jud",
    "revelation": "rev",
}
# Create a lookup for abbreviations to full names
reverse_book_aliases = {v: k for k, v in book_aliases.items()}

In [72]:
def get_normalized_reference(token: str) -> str:
    try:
        book, chapter, verse = token.split('.')
    except ValueError:
        raise ValueError(f"Invalid token format: {token}. Expected format 'book.chapter.verse'.")
    book = book.strip().lower()
    book = reverse_book_aliases[book]
    chapter = chapter.strip()
    verse = verse.strip()
    return f"{book} {chapter}:{verse}"

def get_normalized_reference_df(df: pd.DataFrame, column: str) -> pd.DataFrame:
    df[column] = df[column].apply(get_normalized_reference)
    return df

try_verse_df = get_normalized_reference_df(verse_df, "verse_ref")
try_verse_df.head()

,verse_ref,text
0,1 corinthians 1:1,Παῦλος κλητὸς ἀπόστολος Χριστοῦ Ἰησοῦ διὰ θελή...
1,1 corinthians 1:2,"τῇ ἐκκλησίᾳ τοῦ θεοῦ τῇ οὔσῃ ἐν Κορίνθῳ, ἡγι..."
2,1 corinthians 1:3,χάρις ὑμῖν καὶ εἰρήνη ἀπὸ θεοῦ πατρὸς ἡμῶν καὶ...
3,1 corinthians 1:4,Εὐχαριστῶ τῷ θεῷ μου πάντοτε περὶ ὑμῶν ἐπὶ τῇ...
4,1 corinthians 1:5,ὅτι ἐν παντὶ ἐπλουτίσθητε ἐν αὐτῷ ἐν παντὶ λόγ...


In [73]:
all_verse_df = pd.concat([try_verse_df, ot_df.rename(columns={'reference': 'verse_ref'})], ignore_index=True)
all_verse_df.head()

,verse_ref,text
0,1 corinthians 1:1,Παῦλος κλητὸς ἀπόστολος Χριστοῦ Ἰησοῦ διὰ θελή...
1,1 corinthians 1:2,"τῇ ἐκκλησίᾳ τοῦ θεοῦ τῇ οὔσῃ ἐν Κορίνθῳ, ἡγι..."
2,1 corinthians 1:3,χάρις ὑμῖν καὶ εἰρήνη ἀπὸ θεοῦ πατρὸς ἡμῶν καὶ...
3,1 corinthians 1:4,Εὐχαριστῶ τῷ θεῷ μου πάντοτε περὶ ὑμῶν ἐπὶ τῇ...
4,1 corinthians 1:5,ὅτι ἐν παντὶ ἐπλουτίσθητε ἐν αὐτῷ ἐν παντὶ λόγ...


In [74]:
print(try_verse_df.head())

           verse_ref                                               text
0  1 corinthians 1:1  Παῦλος κλητὸς ἀπόστολος Χριστοῦ Ἰησοῦ διὰ θελή...
1  1 corinthians 1:2  τῇ ἐκκλησίᾳ τοῦ θεοῦ τῇ οὔσῃ ἐν Κορίνθῳ, ἡγι...
2  1 corinthians 1:3  χάρις ὑμῖν καὶ εἰρήνη ἀπὸ θεοῦ πατρὸς ἡμῶν καὶ...
3  1 corinthians 1:4  Εὐχαριστῶ τῷ θεῷ μου πάντοτε περὶ ὑμῶν ἐπὶ τῇ...
4  1 corinthians 1:5  ὅτι ἐν παντὶ ἐπλουτίσθητε ἐν αὐτῷ ἐν παντὶ λόγ...


In [75]:
import pandas as pd

# 1. Standardize casing to ensure matches (optional but recommended)
all_verse_df['verse_ref'] = all_verse_df['verse_ref'].str.lower()
all_pairs_df['Ref_1'] = all_pairs_df['Ref_1'].str.lower()
all_pairs_df['Ref_2'] = all_pairs_df['Ref_2'].str.lower()

# 2. Merge for Ref_1
df_final = all_pairs_df.merge(
    all_verse_df, 
    left_on='Ref_1', 
    right_on='verse_ref', 
    how='left'
).rename(columns={'text': 'text_1'}).drop(columns=['verse_ref'])

# 3. Merge for Ref_2
df_final = df_final.merge(
    all_verse_df, 
    left_on='Ref_2', 
    right_on='verse_ref', 
    how='left'
).rename(columns={'text': 'text_2'}).drop(columns=['verse_ref'])


In [76]:
df_final.head()

,Ref_1,Ref_2,text_1,text_2
0,matthew 1:23,isaiah 7:14,ἰδοὺ ἡ παρθένος ἐν γαστρὶ ἕξει καὶ τέξεται υἱό...,διὰ τοῦτο δώσει κύριος αὐτὸς ὑμῖν σημεῖον· ἰδο...
1,matthew 2:6,micah 5:2,"καὶ σὺ βηθλέεμ γῆ Ἰούδα, οὐδαμῶς ἐλαχίστη εἶ ἐ...",διὰ τοῦτο δώσει αὐτοὺς ἕως καιροῦ τικτούσης τέ...
2,matthew 2:15,hosea 11:1,"καὶ ἦν ἐκεῖ ἕως τῆς τελευτῆς Ἡρῴδου, ἵνα πληρω...","Διότι νήπιος Ισραηλ, καὶ ἐγὼ ἠγάπησα αὐτὸν καὶ..."
3,matthew 2:18,jeremiah 31:15,"φωνὴ ἐν Ῥαμὰ ἠκούσθη, θρῆνος καὶ κλαυθμὸς καὶ ...","ὤλετο Μωαβ πόλις αὐτοῦ, καὶ ἐκλεκτοὶ νεανίσκοι..."
4,matthew 2:18,jeremiah 31:15,"φωνὴ ἐν Ῥαμὰ ἠκούσθη, θρῆνος καὶ κλαυθμὸς καὶ ...",Οὕτως εἶπεν κύριος ὁ θεὸς Ισραηλ Λαβὲ τὸ ποτήρ...


In [78]:
df_final.to_csv("fine_tuning_data/accurate_pairs.csv", index=False)